In [1]:
# BLOK 0: SETUP ENVIRONMENT & DRIVE MOUNTING (GOOGLE COLAB)

import os
import sys
import platform
import pandas as pd
import numpy as np
import sklearn
import joblib

# Cek Versi
print(f"Python Version : {platform.python_version()}")
print(f"Pandas Version : {pd.__version__}")
print(f"NumPy Version  : {np.__version__}")
print(f"Scikit-Learn   : {sklearn.__version__}\n")

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. Buat Folder di Google Drive
PROJECT_DIR = '/content/drive/MyDrive/CAP_DigitalTwin_Transformer/'
SUBDIRS = ['01_Data/Raw', '01_Data/Synthetic', '01_Data/Processed', '02_AI_Engine/Models']

for subdir in SUBDIRS:
    path = os.path.join(PROJECT_DIR, subdir)
    os.makedirs(path, exist_ok=True)

print(f"Done: {PROJECT_DIR}")

Python Version : 3.12.13
Pandas Version : 2.2.2
NumPy Version  : 2.0.2
Scikit-Learn   : 1.6.1



MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Ganti nama kolom data mandarin ke bahasa Inggris agar mudah dipanggil
import pandas as pd
path_xlsx = '/content/drive/MyDrive/CAP_DigitalTwin_Transformer/01_Data/Raw/Data_Mandarin_Lengkap.xlsx'
df_mandarin = pd.read_excel(path_xlsx)
df_mandarin.rename(columns={'故障类型': 'Fault_Type'}, inplace=True)

daftar_penyakit = df_mandarin['Fault_Type'].unique()

print(f"Total kelas terdeteksi: {len(daftar_penyakit)} Kelas")
print("Daftar Kelas:")
for kelas in daftar_penyakit:
    print(f"- {kelas}")

=== MENDETEKSI DAFTAR PENYAKIT MANDARIN ===
Total kelas terdeteksi: 7 Kelas
Daftar Kelas:
- 中温过热
- 高温过热
- 低能放电
- 高能放电
- 正常
- 低温过热
- 局部放电


In [ ]:
import pandas as pd
import numpy as np

# DEFINISI PATH FILE
path_mandarin = '/content/drive/MyDrive/CAP_DigitalTwin_Transformer/01_Data/Raw/Data_Mandarin_Lengkap.xlsx'
path_reference = '/content/drive/MyDrive/CAP_DigitalTwin_Transformer/01_Data/Raw/DGA_Dataset_Reference.csv'

# PROSES DATA 1 - DATASET MANDARIN

print("\n[1/4] Memproses Dataset Mandarin...")
df_mnd = pd.read_excel(path_mandarin)
df_mnd.rename(columns={'故障类型': 'Fault_Type'}, inplace=True)

translation_dict = {
    '正常': 'Normal', '局部放电': 'PD', '低能放电': 'D1', '高能放电': 'D2',
    '低温过热': 'T1', '中温过热': 'T2', '高温过热': 'T3'
}
df_mnd['Label'] = df_mnd['Fault_Type'].map(translation_dict)
df_mnd_clean = df_mnd[['H2', 'CH4', 'C2H6', 'C2H4', 'C2H2', 'Label']].copy()
print(f"Data mandarin ada : {df_mnd_clean.shape[0]} baris.")

# PROSES DATA 2 - DATASET DGA REFERENCE
print("\n[2/4] Memproses Dataset DGA Reference...")
df_ref = pd.read_csv(path_reference)
df_ref.columns = df_ref.columns.str.upper().str.strip()

# Hapus kolom 'NM'
if 'NM' in df_ref.columns:
    df_ref.drop(columns=['NM'], inplace=True)

rename_dict = {}
for col in df_ref.columns:
    if 'C2H2' in col: rename_dict[col] = 'C2H2'
    elif 'C2H4' in col: rename_dict[col] = 'C2H4'
    elif 'C2H6' in col: rename_dict[col] = 'C2H6'
    elif 'CH4' in col: rename_dict[col] = 'CH4'
    elif 'H2' in col and 'C2' not in col: rename_dict[col] = 'H2'
    elif any(x in col for x in ['TARGET', 'LABEL', 'STATUS', 'TYPE', 'FAULT']):
        rename_dict[col] = 'Label'

df_ref.rename(columns=rename_dict, inplace=True)

# Cek jika ada duplikasi kolom setelah rename
df_ref = df_ref.loc[:, ~df_ref.columns.duplicated()]

kolom_wajib = ['H2', 'CH4', 'C2H6', 'C2H4', 'C2H2', 'Label']
df_ref_clean = df_ref[kolom_wajib].copy()

df_ref_clean['Label'] = df_ref_clean['Label'].astype(str).str.title().str.strip()
df_ref_clean['Label'] = df_ref_clean['Label'].replace({'Pd': 'PD'})
df_ref_clean = df_ref_clean[df_ref_clean['Label'].isin(['Normal', 'PD', 'D1', 'D2', 'T1', 'T2', 'T3'])]

print(f"✔️ Dataset DGA Reference berhasil dimuat dan dibersihkan: {df_ref_clean.shape[0]} baris.")


# PENGGABUNGAN DATA
print("\n[3/4] Menggabungkan kedua dataset menjadi satu kesatuan...")
df_combined = pd.concat([df_mnd_clean, df_ref_clean], ignore_index=True)
print(f"✔️ Total data gabungan sebelum difilter: {df_combined.shape[0]} baris.")

# Filter (Standard IEEE dan Spek Trafo)
print("\n[4/4] Menjalankan fungsi penyaringan fisik dan matematis...")

df_combined['Duval_Sum'] = df_combined['CH4'] + df_combined['C2H4'] + df_combined['C2H2'] + 1e-9
df_combined['p_CH4'] = (df_combined['CH4'] / df_combined['Duval_Sum']) * 100
df_combined['p_C2H4'] = (df_combined['C2H4'] / df_combined['Duval_Sum']) * 100
df_combined['p_C2H2'] = (df_combined['C2H2'] / df_combined['Duval_Sum']) * 100

def satpam_filter_ieee_duval(row):
    lbl = str(row['Label']).strip()

    if lbl == 'Normal':
        if (row['H2'] <= 75.0 and row['CH4'] <= 90.0 and
            row['C2H6'] <= 90.0 and row['C2H4'] <= 50.0 and row['C2H2'] <= 1.0):
            return True
        return False

    p_ch4, p_c2h4, p_c2h2 = row['p_CH4'], row['p_C2H4'], row['p_C2H2']

    if lbl == 'PD':
        return p_ch4 >= 98
    elif lbl == 'T1':
        return p_ch4 < 98 and p_c2h4 < 20 and p_c2h2 < 4
    elif lbl == 'T2':
        return p_c2h4 >= 20 and p_c2h4 < 50 and p_c2h2 < 4
    elif lbl == 'T3':
        return p_c2h4 >= 50 and p_c2h2 < 15
    elif lbl == 'D1':
        return p_c2h2 >= 13 and p_c2h4 < 23
    elif lbl == 'D2':
        kondisi_1 = p_c2h4 >= 23 and p_c2h2 >= 29
        kondisi_2 = (p_c2h4 >= 23 and p_c2h4 < 40) and (p_c2h2 >= 13 and p_c2h2 < 29)
        return kondisi_1 or kondisi_2

    return False

df_combined['Is_Valid'] = df_combined.apply(satpam_filter_ieee_duval, axis=1)

df_premium_final = df_combined[df_combined['Is_Valid'] == True].copy()
df_rejected = df_combined[df_combined['Is_Valid'] == False].copy()

df_premium_final = df_premium_final[['H2', 'CH4', 'C2H6', 'C2H4', 'C2H2', 'Label']]


print(f"Hasil Filter:")
print(f"- Data Melanggar Standar Dibuang : {df_rejected.shape[0]} baris")
print(f"- Data Bersih : {df_premium_final.shape[0]} baris")

path_output_final = '/content/drive/MyDrive/CAP_DigitalTwin_Transformer/01_Data/Raw/Data_Diagnosis_Premium_Gabungan.csv'
df_premium_final.to_csv(path_output_final, index=False)
print(f"Sukses disimpan di: {path_output_final}")

⏳ Memulai Ulang Proses Integrasi Jalur A (Perbaikan Ganda pada Kolom)...

[1/4] Memproses Dataset Mandarin...
✔️ Dataset Mandarin berhasil dimuat: 2321 baris.

[2/4] Memproses Dataset DGA Reference...
✔️ Dataset DGA Reference berhasil dimuat dan dibersihkan: 0 baris.

[3/4] Menggabungkan kedua dataset menjadi satu kesatuan...
✔️ Total data gabungan sebelum difilter: 2321 baris.

[4/4] Menjalankan fungsi penyaringan fisik dan matematis...

📊 HASIL AKHIR PENYARINGAN JALUR A:
- Data Cacat/Melanggar Standar Dibuang : 812 baris
- Data Bersih & Premium Siap untuk AI  : 1509 baris

📁 File premium final sukses disimpan di: /content/drive/MyDrive/CAP_DigitalTwin_Transformer/01_Data/Raw/Data_Diagnosis_Premium_Gabungan.csv


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os
import warnings
warnings.filterwarnings('ignore')


# 1. Memuat Dataset Gabungan
path_diagnosis = '/content/drive/MyDrive/CAP_DigitalTwin_Transformer/01_Data/Raw/Data_Diagnosis_Premium_Gabungan.csv'
if not os.path.exists(path_diagnosis):
    raise FileNotFoundError(f"File tidak ditemukan di {path_diagnosis}. Pastikan Jalur A sudah dijalankan.")

df_diag = pd.read_csv(path_diagnosis)

# Pemisahan Fitur (X) dan Target Label (y)
X_diag = df_diag[['H2', 'CH4', 'C2H6', 'C2H4', 'C2H2']]
y_diag = df_diag['Label']

# Stratified Split (80% Training, 20% Testing)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diag, y_diag, test_size=0.2, random_state=42, stratify=y_diag
)

# Inisialisasi & Pelatihan Model Random Forest
model_diagnosis = RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced', max_depth=10)
model_diagnosis.fit(X_train_d, y_train_d)

# Evaluasi Kinerja pada Data Uji (20%)
y_pred_d = model_diagnosis.predict(X_test_d)
akurasi_d = accuracy_score(y_test_d, y_pred_d)

print(f"Akurasi Model: {akurasi_d * 100:.2f}%")
print("Performa PRediksi Tiap Class:")
print(classification_report(y_test_d, y_pred_d))

# Buat File (.pkl) ke Google Drive
path_dir_model = '/content/drive/MyDrive/CAP_DigitalTwin_Transformer/02_AI_Engine/Models'
os.makedirs(path_dir_model, exist_ok=True)
path_export_diag = os.path.join(path_dir_model, 'model_dga_7classes_v2.pkl')

joblib.dump(model_diagnosis, path_export_diag)
print(f"sukses disimpan di: {path_export_diag}")

⏳ [1/2] Memulai Pelatihan Otak 1: AI Diagnosis Penyakit DGA...

🏆 BERHASIL! AKURASI MODEL DIAGNOSIS: 94.37%

📊 LAPORAN PERFORMA PREDIKSI TIAP KELAS:
              precision    recall  f1-score   support

          D1       0.95      0.89      0.92        47
          D2       0.89      0.95      0.92        41
      Normal       0.98      0.97      0.98       141
          PD       1.00      1.00      1.00         5
          T1       0.86      0.75      0.80         8
          T2       0.55      0.67      0.60         9
          T3       0.98      0.98      0.98        51

    accuracy                           0.94       302
   macro avg       0.89      0.89      0.89       302
weighted avg       0.95      0.94      0.94       302

📁 Otak 1 sukses disimpan di: /content/drive/MyDrive/CAP_DigitalTwin_Transformer/02_AI_Engine/Models/model_dga_7classes_v2.pkl
